In [1]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

/data/2_data_server/cv-07/anaconda3/envs/caption_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# GPU 사용 가능 여부 확인 및 장치 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 장치: {device}")

# Hugging Face에서 사전 학습된 모델과 프로세서 로드
# 모델을 처음 실행하면 캐시 폴더에 다운로드되며, 약간의 시간이 소요될 수 있습니다.
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


사용 장치: cuda


In [4]:
# 캡셔닝을 진행할 이미지 경로
# !!! 아래 파일명을 실제 본인의 이미지 파일 경로로 변경해주세요 !!!
image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_001.jpg" # 예시: "C:/Users/YourName/Pictures/cat_on_sofa.png"

try:
    # 이미지 열기
    raw_image = Image.open(image_path).convert('RGB')
    print(f"'{image_path}' 이미지를 성공적으로 불러왔습니다.")

    # 1. 이미지 전처리 (Processor 활용)
    # 텍스트 프롬프트 없이 이미지만으로 캡셔닝을 생성합니다.
    inputs = processor(images=raw_image, return_tensors="pt").to(device)

    # 2. 캡션 생성 (Model 활용)
    # `generate` 함수는 이미지에 대한 텍스트 설명을 생성합니다.
    out = model.generate(**inputs, max_length=50) # max_length로 최대 길이를 조절할 수 있습니다.

    # 3. 결과 후처리 (Processor 활용)
    # 생성된 텍스트 ID를 사람이 읽을 수 있는 문장으로 디코딩합니다.
    caption = processor.decode(out[0], skip_special_tokens=True)

    # 생성된 캡션 출력
    print("-" * 50)
    print("생성된 캡션:")
    print(caption)
    print("-" * 50)

except FileNotFoundError:
    print(f"오류: '{image_path}' 경로에서 이미지를 찾을 수 없습니다.")
    print("파일명이나 경로가 올바른지 다시 확인해주세요.")
except Exception as e:
    print(f"오류가 발생했습니다: {e}")

'/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_001.jpg' 이미지를 성공적으로 불러왔습니다.
--------------------------------------------------
생성된 캡션:
people are running on tread machines in a gym with bright orange walls
--------------------------------------------------
